In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')


In [3]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)


(array([0, 1]), array([900, 100]))

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


In [5]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)


(array([0, 1]), array([619, 619]))

In [6]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'lbfgs'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]


In [7]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)


In [8]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost


In [9]:


# Initialize MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Set tracking URI first
mlflow.set_tracking_uri("http://localhost:5000")

# Set experiment
mlflow.set_experiment("deployment model")

for i, element in enumerate(models):

    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })

        # Log models
        if "XGB" in model_name:
            mlflow.xgboost.log_model(
                model,
                artifact_path="model"
            )

        else:
            mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                serialization_format="cloudpickle"
            )

print("All models logged successfully!")


2026/11/01 01:02:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/01 01:02:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/01 01:02:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/4/runs/97a84b7383c04909addb29c144dfccc4
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/11/01 01:02:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/01 01:04:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/4/runs/c6a07faa8e7441bdae2e544b56419b60
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/11/01 01:04:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/4/runs/d8df216de3174298acee0337b2d41079
🧪 View experiment at: http://localhost:5000/#/experiments/4
🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/4/runs/4badba5a422347bf8a1649cdd3e4d9f7
🧪 View experiment at: http://localhost:5000/#/experiments/4
All models logged successfully!


In [12]:
import mlflow

model_name = "XGB-Smote"

run_id = "fac657416c4349bbbb459d61344cf9ce"

model_uri = f"runs:/{run_id}/model"

with mlflow.start_run(run_id=run_id):
    result = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

print("Model registered successfully!")
print("Model Name:", model_name)
print("Model Version:", result.version)


Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/11/01 01:10:08 WARNING mlflow.tracking._model_registry.fluent: Run with id fac657416c4349bbbb459d61344cf9ce has no artifacts at artifact path 'model', registering model based on models:/m-39f285894f3c46bc86a914ed88501398 instead
2026/11/01 01:10:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/4/runs/fac657416c4349bbbb459d61344cf9ce
🧪 View experiment at: http://localhost:5000/#/experiments/4
Model registered successfully!
Model Name: XGB-Smote
Model Version: 1


Created version '1' of model 'XGB-Smote'.


In [13]:
model_name = 'XGB-Smote'
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]


array([0, 0, 0, 0])

In [16]:
import mlflow

client = mlflow.MlflowClient()

versions = client.search_model_versions("name='XGB-Smote'")

for version in versions:
    print("Model:", version.name)
    print("Version:", version.version)
    print("Run ID:", version.run_id)
    print("-------------------")

Model: XGB-Smote
Version: 1
Run ID: fac657416c4349bbbb459d61344cf9ce
-------------------


In [20]:
import mlflow

client = mlflow.MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())

Tracking URI: http://localhost:5000
Registry URI: http://localhost:5000


In [21]:
models = client.search_registered_models()

for model in models:
    print("Model Name:", model.name)

Model Name: XGB-Smote


In [23]:
versions = client.search_model_versions(
    "name='XGB-Smote'"
)

for v in versions:
    print("Model:", v.name)
    print("Version:", v.version)
    print("Status:", v.status)
    print("-------------------")

Model: XGB-Smote
Version: 1
Status: READY
-------------------


In [24]:
import mlflow

client = mlflow.MlflowClient()

versions = client.search_model_versions("name='XGB-Smote'")

for v in versions:
    print("Version:", v.version)
    print("Run ID:", v.run_id)
    print("Status:", v.status)

Version: 1
Run ID: fac657416c4349bbbb459d61344cf9ce
Status: READY


In [25]:
source_version = client.get_model_version(
    name="XGB-Smote",
    version="1"
)

run_id = source_version.run_id

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(
    model_uri=model_uri,
    name="XGB-Smote"
)

print("New Version:", result.version)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/11/01 01:21:16 WARNING mlflow.tracking._model_registry.fluent: Run with id fac657416c4349bbbb459d61344cf9ce has no artifacts at artifact path 'model', registering model based on models:/m-39f285894f3c46bc86a914ed88501398 instead
2026/11/01 01:21:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 2


New Version: 2


Created version '2' of model 'XGB-Smote'.


In [26]:
version5 = client.get_model_version(
    name="XGB-Smote",
    version="5"
)

print("Model:", version5.name)
print("Version:", version5.version)
print("Status:", version5.status)

RestException: RESOURCE_DOES_NOT_EXIST: Model Version (name=XGB-Smote, version=5) not found

In [27]:
import mlflow

client = mlflow.MlflowClient()

versions = client.search_model_versions("name='XGB-Smote'")

for v in versions:
    print("Version:", v.version)
    print("Run ID:", v.run_id)
    print("Status:", v.status)
    print("Source:", v.source)

Version: 2
Run ID: fac657416c4349bbbb459d61344cf9ce
Status: READY
Source: models:/m-39f285894f3c46bc86a914ed88501398
Version: 1
Run ID: fac657416c4349bbbb459d61344cf9ce
Status: READY
Source: models:/m-39f285894f3c46bc86a914ed88501398


In [28]:
source_version = client.get_model_version(
    name="XGB-Smote",
    version="1"
)

model_uri = source_version.source

result = mlflow.register_model(
    model_uri=model_uri,
    name="XGB-Smote"
)

print("New Version:", result.version)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/11/01 01:23:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 3


New Version: 3


Created version '3' of model 'XGB-Smote'.


In [30]:
result = mlflow.register_model(
    model_uri=source_version.source,
    name="XGB-Smote"
)

print("Created Version:", result.version)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/11/01 01:24:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 4


Created Version: 4


Created version '4' of model 'XGB-Smote'.


In [31]:
result = mlflow.register_model(
    model_uri=source_version.source,
    name="XGB-Smote"
)

print("Created Version:", result.version)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/11/01 01:24:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 5


Created Version: 5


Created version '5' of model 'XGB-Smote'.


In [32]:
versions = client.search_model_versions(
    "name='XGB-Smote'"
)

for v in versions:
    print("Version:", v.version, "Status:", v.status)

Version: 5 Status: READY
Version: 4 Status: READY
Version: 3 Status: READY
Version: 2 Status: READY
Version: 1 Status: READY


In [33]:
result = client.copy_model_version(
    src_model_uri="models:/XGB-Smote/5",
    dst_name="finalproduction"
)

print("Copied Version:", result.version)

Copied Version: 1


Successfully registered model 'finalproduction'.
Copied version '5' of model 'XGB-Smote' to version '1' of model 'finalproduction'.


In [34]:
'''
# ignore below code

model_version = 2
prod_model_uri = f"models:/{production_model_name}@productionserver2"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]
'''


'\n# ignore below code\n\nmodel_version = 2\nprod_model_uri = f"models:/{production_model_name}@productionserver2"\n\nloaded_model = mlflow.xgboost.load_model(prod_model_uri)\ny_pred = loaded_model.predict(X_test)\ny_pred[:4]\n'